In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import display
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
TASK = Path.cwd().parent
ROOT = TASK.parent
INPUT, OUTPUT = TASK/'input', TASK/'output'
CONFIG = json.loads((ROOT/'config.json').read_text())
EPS, Y, SEED = CONFIG['eps'], CONFIG['last_year'], CONFIG['seed']

In [ ]:
sns.set_theme(style='whitegrid',context='notebook',palette='colorblind')
SELECTED = CONFIG['models']
ORDER = ['empirical_complete'] + SELECTED
PALETTE = dict(zip(ORDER,sns.color_palette('colorblind',len(ORDER))))
DASHES = dict(zip(ORDER,['',(4,2),(1,1),(6,2,1,2),(3,1,1,1),(8,2)]))
MARKERS = dict(zip(ORDER,['o','s','^','D','v','P']))
def save(fig,name):
    fig.tight_layout()
    fig.savefig(OUTPUT/(name+'.png'),dpi=180,bbox_inches='tight')
    fig.savefig(OUTPUT/(name+'.pdf'),bbox_inches='tight')
    plt.show()
    plt.close(fig)
def lines(data,x,y,ax):
    sns.lineplot(data=data,x=x,y=y,hue='source',style='source',hue_order=ORDER,style_order=ORDER,
                 palette=PALETTE,dashes=DASHES,markers=MARKERS,markersize=3,errorbar=None,ax=ax)
def legend(ax):
    ax.legend(fontsize=7,ncol=2)
def cdf_lines(data,x,y,ax):
    # duplicate x values draw vertical jumps; no averaging of left/right ECDF limits
    for source in ORDER:
        d = data.loc[data.source.eq(source)].sort_values(x,kind='stable')
        sns.lineplot(data=d,x=x,y=y,estimator=None,sort=False,color=PALETTE[source],
                     linestyle='-' if not DASHES[source] else (0,DASHES[source]),label=source,ax=ax)
    ax.set_ylim(-.02,1.02)

In [ ]:
cdf = pd.read_csv(INPUT/'distribution_cdf.csv')
increments = pd.read_csv(INPUT/'increment_cdf.csv',dtype={'stage':str})
# saved knots retain exact left/right probabilities; draw both sides of each saved jump
cdf = pd.concat([cdf.assign(probability=cdf.cdf_left,side=0),cdf.assign(probability=cdf.cdf,side=1)],ignore_index=True).sort_values(['source','diagnostic','x','side'])
increments = pd.concat([increments.assign(probability=increments.cdf_left,side=0),increments.assign(probability=increments.cdf,side=1)],ignore_index=True).sort_values(['source','stage','scale','x','side'])

In [ ]:
# full observations for exact ECDF figures; the CSVs retain only compact evaluation knots
panel = pd.read_csv(INPUT/'simulation/fit/data/panel.csv',index_col='dblp_id')
Q = panel.to_numpy(dtype=float).T
sources = {'empirical_complete':Q[:,np.isfinite(Q).all(axis=0)],
           **{tag:np.load(INPUT/'simulation'/tag/'trajectories.npy') for tag in SELECTED}}
samples = {}
windows = {'stage_0':(0,0),'stage_1_4':(1,4),'stage_5_7':(5,7),'stage_8_20':(8,Y),
           'through_5':(0,5),'last_four':(Y-3,Y),'career_total':(0,Y),'generated_common':(6,Y)}
for source,array in sources.items():
    samples[source] = {key:array[start:end+1].sum(axis=0) for key,(start,end) in windows.items()}
    samples[source].update({'annual_pooled':array.ravel(),'career_sd':array.std(axis=0),
                            'career_sd_positive':np.array([v[v>0].std() if (v>0).sum()>1 else np.nan for v in array.T]),
                            'zero_count':np.sum(array==0,axis=0),'year_max':array.argmax(axis=0)})
    for t in [0,4,7,13,Y]:samples[source][f'annual_{t}'] = array[t]
def cdf_lines(data,x,y,ax):
    for source in ORDER:
        if 'diagnostic' in data.columns:
            values = samples[source][data.diagnostic.iloc[0]]
        else:
            row = data.iloc[0]
            start,end = {'pooled':(0,Y-1),'0':(0,0),'1-4':(1,4),'5-7':(5,7),'8-20':(8,Y-1)}[row.stage]
            array = sources[source]
            delta = array[1:]-array[:-1] if row.scale=='raw' else np.log(array[1:]+EPS)-np.log(array[:-1]+EPS)
            values = delta[start:end+1].ravel()
        values = values[np.isfinite(values)]
        sns.ecdfplot(x=values,color=PALETTE[source],linestyle='-' if not DASHES[source] else (0,DASHES[source]),
                     rasterized=True,label=source,ax=ax)
    ax.legend()
    ax.set_ylim(-.02,1.02)

In [ ]:
groups = {'annual':['annual_0','annual_4','annual_7','annual_13',f'annual_{Y}','annual_pooled'],
          'totals':['through_5','last_four','career_total','generated_common','stage_1_4','stage_8_20'],
          'career':['career_sd','career_sd_positive','zero_count','year_max','stage_0','stage_5_7']}
for name,diagnostics in groups.items():
    fig,axes = plt.subplots(2,3,figsize=(14,8))
    for ax,diagnostic in zip(axes.flat,diagnostics):
        cdf_lines(cdf.loc[cdf.diagnostic.eq(diagnostic)],'x','probability',ax)
        if diagnostic not in ['zero_count','year_max']:ax.set_xscale('symlog',linthresh=1)
        ax.set(title=diagnostic.replace('_',' '),xlabel='Productivity / statistic',ylabel='Cumulative probability')
        if ax!=axes.flat[0]:ax.legend().remove()
    legend(axes.flat[0])
    fig.suptitle('Empirical ECDF / simulated CDF; complete empirical careers')
    save(fig,f'cdf_{name}')

In [ ]:
for scale in ['raw','offset_log']:
    fig,axes = plt.subplots(2,3,figsize=(14,8))
    for ax,stage in zip(axes.flat,['pooled','0','1-4','5-7','8-20']):
        cdf_lines(increments.loc[increments.scale.eq(scale)&increments.stage.eq(stage)],'x','probability',ax)
        if scale=='raw':ax.set_xscale('symlog',linthresh=1)
        ax.set(title=stage,xlabel='Increment',ylabel='Cumulative probability')
        if ax!=axes.flat[0]:ax.legend().remove()
    axes.flat[-1].axis('off')
    legend(axes.flat[0])
    fig.suptitle(f'Increment CDFs — {scale}; full transition windows include AR(6,6) prefix')
    save(fig,f'increment_cdf_{scale}')

In [ ]:
# empirical-versus-model views use the same complete empirical cohort for every model
for tag in SELECTED:
    saved_order = ORDER
    ORDER = ['empirical_complete',tag]
    fig,axes = plt.subplots(2,3,figsize=(13,8))
    for ax,diagnostic in zip(axes.flat,['annual_pooled','through_5','last_four','career_total','career_sd','zero_count']):
        cdf_lines(cdf.loc[cdf.diagnostic.eq(diagnostic)],'x','probability',ax)
        if diagnostic!='zero_count':ax.set_xscale('symlog',linthresh=1)
        ax.set(title=diagnostic.replace('_',' '),xlabel='Productivity / statistic',ylabel='Cumulative probability')
        if ax!=axes.flat[0]:ax.legend().remove()
    legend(axes.flat[0])
    fig.suptitle(f'Empirical ECDF vs simulated CDF — {tag}')
    save(fig,f'ecdf_vs_simcdf_{tag}')
    ORDER = saved_order